# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# LangChain 0.3 supports Python 3.13 (0.2.x needs NumPy 1.x, which has no Python 3.13 wheels)
!pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-openai>=0.2,<0.4"

Defaulting to user installation because normal site-packages is not writeable
  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 10.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 15.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 13.6 MB/s eta 0:00:00
Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl (940 kB)

  Attempting uninstall: packaging

    Found existing installation: packaging 26.3

    Uninstalling packaging-26.3:

      Successfully uninstalled packaging-26.3

   ----------------------------------------  0/12 [packaging]
   ------------- --------------------------  4/12 [marshmallow]
  Attempting uninstall: openai
   ------------- -----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.11.2 requires pillow>=9, which is not installed.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [2]:
!pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [3]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [41]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration

class ChatOpenAINoStop(ChatOpenAI):
    """For models that reject 'stop': don't send it, cut the reply ourselves."""
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        result = super()._generate(messages, stop=None, run_manager=run_manager, **kwargs)
        if stop:
            generations = []
            for gen in result.generations:
                text = gen.message.content
                for s in stop:
                    text = text.split(s)[0]
                generations.append(ChatGeneration(message=AIMessage(content=text)))
            result.generations = generations
        return result

llm = ChatOpenAINoStop(model="gpt-6-luna", temperature=1, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [42]:
doc_path =  "./datasets/state_of_the_union.txt"

In [43]:
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [44]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [45]:
from langchain_community.document_loaders import WebBaseLoader

In [46]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [47]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [48]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [49]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [50]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [51]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
Action: State of Union QA System  
Action Input: What did President Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: President Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court four days earlier, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”
Thought:Final Answer: Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court four days earlier, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court four days earlier, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”'}

In [52]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
Question: Why use ruff over flake8?  
Thought: I should check the Ruff QA system for a grounded comparison.  
Action: Ruff QA System  
Action Input: Why use Ruff over Flake8? What are Ruff’s advantages and trade-offs?  
Observation: **Ruff is appealing if you want a fast, consolidated linter**: it reimplements many popular Flake8 plugins, so you can often replace Flake8 plus several extensions with one tool. Ruff also supports automatically fixing many violations, and its rule prefixes make it easier to enable or disable whole rule groups without code conflicts.

**Trade-offs:**

- **It isn’t a complete Flake8 replacement.** Ruff doesn’t support custom lint rules or third-party plugins, and it doesn’t include every rule from every Flake8 plugin—for example, it omits some opinionated flake8-bugbear rules.
- **Rule coverage and behavior differ.** Ruff may use different rule codes and can produce different results from the original plugin. Check you

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff is a good choice if you want a fast linter that combines many rules from Flake8 and its plugins in one tool, with convenient autofixes. It may not be a full replacement: it lacks custom-rule and third-party plugin support, and its rule coverage and behavior can differ. If you rely on specific Flake8 plugins or custom rules, check compatibility before switching.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [53]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [54]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [55]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
Question: What did biden say about ketanji brown jackson in the state of the union address?  
Thought: I should check the State of the Union address for Biden’s remarks about Ketanji Brown Jackson.  
Action: State of Union QA System  
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?  
Observation: Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”'}

In [56]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
Thought: I should consult the Ruff QA system for a grounded comparison.
Action: Ruff QA System
Action Input: Why use Ruff over Flake8?
Observation: Use Ruff if you want a fast, unified alternative to Flake8 and many of its popular plugins. Ruff implements those rules natively, offers a large rule set, and can automatically fix many violations—so you can often replace several tools with one.

It isn’t a perfect drop-in replacement: Ruff doesn’t support custom lint rules and omits some opinionated flake8-bugbear rules. Check that its enabled rules match your project’s needs.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': 'Use Ruff if you want a fast, unified alternative to Flake8 and many of its popular plugins. Ruff implements those rules natively, offers a large rule set, and can automatically fix many violations—so you can often replace several tools with one.\n\nIt isn’t a perfect drop-in replacement: Ruff doesn’t support custom lint rules and omits some opinionated flake8-bugbear rules. Check that its enabled rules match your project’s needs.'}

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [57]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [58]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [59]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
Thought: I should check Ruff’s documentation for how it processes Jupyter Notebooks, then check the State of the Union address for whether the president mentioned that tool.  
Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with **nbQA**, which can run Ruff over Jupyter Notebooks. For example: `nbqa ruff Untitled.ipynb`.
Thought:Thought: I should check the State of the Union address for a mention of nbQA.  
Action: State of Union QA System  
Action Input: Did the president mention nbQA, the tool that can run Ruff over Jupyter Notebooks, in the most recent State of the Union address?  
Observation: No—not in the passage provided. It discusses infrastructure and doesn’t mention nbQA. I can’t verify the full most recent State of the Union from this excerpt alone.
Thought:Thought: Ruff uses nbQA to run over Jupyter Notebooks. The State of the Union result says nbQA was not m

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses **nbQA** to run over Jupyter Notebooks. The available State of the Union result does not mention nbQA, though it cannot confirm whether the tool appears anywhere in the full address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



<br>

## My solution: Shakespeare's Sonnets + Ruff

For my dataset I chose Shakespeare's **Sonnets** (`sonnets.txt` from the suggested repository). It pairs well with the Ruff docs because the two sources have nothing in common, so it's easy to see whether the agent routes each question to the right tool.

### 1. Get the data

Download `sonnets.txt` into the `datasets` folder (skipped if the file is already there).

In [60]:
import urllib.request

sonnets_path = "./datasets/sonnets.txt"
sonnets_url = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"

if not os.path.exists(sonnets_path):
    os.makedirs(os.path.dirname(sonnets_path), exist_ok=True)
    urllib.request.urlretrieve(sonnets_url, sonnets_path)

with open(sonnets_path, encoding="utf-8") as f:
    print(f.read()[:600])

	SONNETS



TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light'st flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And 


### 2. Rebuild the vector store

Same steps as the demo: load, split, embed, and store in a new Chroma collection, then wrap it in a `RetrievalQA` chain.

In [61]:
sonnets_loader = TextLoader(sonnets_path, encoding="utf-8")
sonnets_documents = sonnets_loader.load()
sonnets_texts = text_splitter.split_documents(sonnets_documents)
print(f"{len(sonnets_texts)} chunks")

sonnets_db = Chroma.from_documents(sonnets_texts, embeddings, collection_name="shakespeare-sonnets")

sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

154 chunks


### 3. Rewrite the tool description & 4. Rebuild the agent

The new tool replaces the state of the union one. Its description tells the agent to use it for anything about Shakespeare's sonnets or poetry; Ruff questions still go to the Ruff tool.

In [76]:
sonnets_tool_description = (
    "useful for when you need to answer questions about Shakespeare's sonnets: "
    "their themes (love, beauty, time, aging, death, poetry), imagery and what the "
    "poet says to the young man or the dark lady. Input should be a fully formed "
    "question, not referencing any obscure pronouns from the conversation before."
)
ruff_tool_description = (
    "useful for when you need to answer questions about ruff (a python linter). "
    "Input should be a fully formed question, not referencing any obscure pronouns "
    "from the conversation before."
)

tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description=sonnets_tool_description,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description=ruff_tool_description,
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True, handle_parsing_errors=True
)

### 5. Put it to the test

**A question only the sonnets can answer:**

In [68]:
agent.invoke(
    "According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?"
)



> Entering new AgentExecutor chain...
Question: According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?  
Thought: I should consult the Shakespeare sonnets QA system.  
Action: Shakespeare Sonnets QA System  
Action Input: According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?  
Observation: The beloved’s beauty will live on in the poet’s verse—preserved in “black ink” and remembered by future readers even after time destroys it.
Thought:Final Answer: The beloved’s beauty will live on in Shakespeare’s verse, preserved in words and remembered by future readers after time destroys the original.

> Finished chain.


{'input': "According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?",
 'output': 'The beloved’s beauty will live on in Shakespeare’s verse, preserved in words and remembered by future readers after time destroys the original.'}

**A question only Ruff can answer:**

In [69]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
Action: Ruff QA System  
Action Input: Why use Ruff over Flake8?  
Observation: Ruff can replace Flake8 plus many popular plugins, with those checks built in rather than installed and managed separately. Its rule prefixes are also organized so you can enable groups consistently, and it can automatically fix some violations.

The tradeoff: Ruff doesn’t support custom lint rules, and it doesn’t include every opinionated `flake8-bugbear` rule. If you rely on custom Flake8 plugins or those missing checks, you may need to keep additional tooling.
Thought:Final Answer: Ruff is often preferred over Flake8 because it can replace Flake8 and many popular plugins, with checks built in and automatic fixes for some violations. It’s typically simpler to configure and manage. The tradeoff is that Ruff doesn’t support custom lint rules and may not include every check from plugins such as `flake8-bugbear`, so projects relying on those may need additional tools.



{'input': 'Why use ruff over flake8?',
 'output': 'Ruff is often preferred over Flake8 because it can replace Flake8 and many popular plugins, with checks built in and automatic fixes for some violations. It’s typically simpler to configure and manage. The tradeoff is that Ruff doesn’t support custom lint rules and may not include every check from plugins such as `flake8-bugbear`, so projects relying on those may need additional tools.'}

**A multi-hop question that needs both tools** (same shape as the Jupyter/Ruff example in the demo):

In [70]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?"
)



> Entering new AgentExecutor chain...
Question: What tool does ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?  
Thought: I should check Ruff’s documentation for its Jupyter Notebook support, then check whether that tool appears in Shakespeare’s sonnets.  
Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff can run over Jupyter Notebooks using **nbQA**. After installing both tools, run `nbqa ruff notebook.ipynb`.
Thought:Question: What tool does Ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?  
Thought: Ruff uses nbQA; I should check whether Shakespeare mentions nbQA in his sonnets.  
Action: Shakespeare Sonnets QA System  
Action Input: Does Shakespeare mention nbQA in any of his sonnets?  
Observation: No. “nbQA” does not appear in the sonnets provided.
Thought:Thought: I now know the final answer.  
Final Answer: Ruff uses **nbQA** to ru

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?',
 'output': 'Ruff uses **nbQA** to run Ruff over Jupyter Notebooks (for example, `nbqa ruff notebook.ipynb`). Shakespeare does not mention nbQA in his sonnets.'}

**A second multi-hop question**, where both tools contribute a real answer:

In [71]:
agent.invoke(
    "How much faster does Ruff claim to be than existing Python linters? "
    "And what does Shakespeare say about the passing of time in his sonnets?"
)



> Entering new AgentExecutor chain...
Question: How much faster does Ruff claim to be than existing Python linters? And what does Shakespeare say about the passing of time in his sonnets?  
Thought: I’ll check Ruff’s stated performance claim and Shakespeare’s treatment of time in the sonnets.  
Action: Ruff QA System  
Action Input: How much faster does Ruff claim to be than existing Python linters?  
Observation: Ruff claims to be **10–100× faster** than existing Python linters, such as Flake8.
Thought:Question: How much faster does Ruff claim to be than existing Python linters? And what does Shakespeare say about the passing of time in his sonnets?  
Thought: I’ve got Ruff’s performance claim. I’ll check what the sonnets say about time.  
Action: Shakespeare Sonnets QA System  
Action Input: What do Shakespeare’s sonnets say about the passing of time?  
Observation: Shakespeare’s sonnets portray time as relentless: moments hurry toward their end, youth and beauty fade, and even nat

{'input': 'How much faster does Ruff claim to be than existing Python linters? And what does Shakespeare say about the passing of time in his sonnets?',
 'output': 'Ruff claims to be **10–100× faster** than existing Python linters. In Shakespeare’s sonnets, time is relentless: it wears down youth and beauty and causes things to decay. The speaker hopes to resist that loss through poetry and, in some sonnets, through future generations.'}

### Router mode: `return_direct=True`

The same tools, but each returns the `RetrievalQA` answer directly, with no further reasoning by the agent.

In [77]:
router_tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description=sonnets_tool_description,
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description=ruff_tool_description,
        return_direct=True,
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True, handle_parsing_errors=True
)

In [78]:
router_agent.invoke(
    "According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?"
)



> Entering new AgentExecutor chain...
Question: According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?  
Thought: I should consult the Shakespeare sonnets QA system.  
Action: Shakespeare Sonnets QA System  
Action Input: According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?  
Observation: The beloved’s beauty will live on in the poet’s written verse—preserved in “black ink” and remembered by readers.


> Finished chain.


{'input': "According to Shakespeare's sonnets, what will keep the beloved's beauty alive after time destroys it?",
 'output': 'The beloved’s beauty will live on in the poet’s written verse—preserved in “black ink” and remembered by readers.'}

In [79]:
router_agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
Question: Why use ruff over flake8?  
Thought: I should consult the Ruff QA system for a comparison.  
Action: Ruff QA System  
Action Input: Why use Ruff over Flake8?  
Observation: Ruff can replace Flake8 plus many popular plugins in one tool: their rules are built in, and Ruff can automatically fix some violations. It also groups rules under consistent prefixes, making them easier to enable or disable.

The trade-offs: Ruff doesn’t support custom or third-party lint rules, and it doesn’t include every rule from every plugin—for example, some opinionated flake8-bugbear rules are missing. Check your existing Flake8 configuration before switching.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': 'Ruff can replace Flake8 plus many popular plugins in one tool: their rules are built in, and Ruff can automatically fix some violations. It also groups rules under consistent prefixes, making them easier to enable or disable.\n\nThe trade-offs: Ruff doesn’t support custom or third-party lint rules, and it doesn’t include every rule from every plugin—for example, some opinionated flake8-bugbear rules are missing. Check your existing Flake8 configuration before switching.'}

In [80]:
# The multi-hop question again, in router mode
router_agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?"
)



> Entering new AgentExecutor chain...
Thought: I should check what Ruff uses for notebook support, then verify whether that tool appears in Shakespeare’s sonnets.
Action: Ruff QA System
Action Input: What tool does Ruff use to run over Jupyter Notebooks?
Observation: Ruff integrates with **nbQA**, which you can use to run Ruff over Jupyter Notebooks. For example: `nbqa ruff Untitled.ipynb`.


> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did Shakespeare mention that tool in his sonnets?',
 'output': 'Ruff integrates with **nbQA**, which you can use to run Ruff over Jupyter Notebooks. For example: `nbqa ruff Untitled.ipynb`.'}

### 6. Reflect

**Did the agent pick the right tool(s)?** Yes, every time. In normal mode (agent as reasoner):

| Question | Tool(s) called | Result |
|---|---|---|
| What keeps the beloved's beauty alive? | Shakespeare Sonnets QA | ✅ "The poet's verse, preserved in black ink" (Sonnet 65) |
| Why use Ruff over Flake8? | Ruff QA | ✅ One tool replacing Flake8 + plugins, autofixes; no custom rules |
| Ruff's Jupyter tool + did Shakespeare mention it? | Ruff QA → Shakespeare Sonnets QA | ✅ nbQA; not in the sonnets |
| Ruff's speed + Shakespeare on time | Ruff QA → Shakespeare Sonnets QA | ✅ 10–100× faster; time is relentless, resisted through children and poetry |

- The tool descriptions were enough for routing. Questions naming Shakespeare or sonnets went straight to the sonnets tool, and Ruff questions to the Ruff tool, with no wrong first picks.
- In the multi-hop questions the agent did real chaining. It took the answer from the first tool (*nbQA*) and wrote a new, self-contained question for the second tool ("Does Shakespeare mention nbQA in any of his sonnets?"), as the tool description asks ("no obscure pronouns"). Its *Final Answer* then combined both observations.
- One limitation: the sonnets tool answered "not in the sonnets **provided**". `RetrievalQA` only sees the top 4 retrieved chunks, not all 154, so a "not mentioned" answer only covers what was retrieved. The demo's State of the Union tool hedged the same way about nbQA.

**`return_direct=True` vs. not:**

- **Single-source questions:** the same tool was picked and the answer was essentially the same. The log shows the chain ending right after the `Observation` with an empty step: the agent never writes a `Final Answer`, so the `RetrievalQA` output is returned verbatim. That saves one LLM call per question. Without `return_direct`, the agent rephrased the observation into its own final answer, which added little for these questions.
- **Multi-hop question:** router mode broke it. The agent's first thought even planned to check Ruff *and then* the sonnets, but as soon as the Ruff tool returned, `return_direct` ended the chain. The output was only "nbQA…" and the Shakespeare half of the question was never answered.
- **Conclusion:** router mode is faster and cheaper when a question belongs to exactly one source; multi-hop questions need the agent to keep reasoning after each tool call.

**Note on the model:** OpenAI shut down the lab's default model (`gpt-3.5-turbo-instruct`), so I used `gpt-6-luna`. It rejects the `stop` parameter the ReAct agent relies on, which needed a small wrapper class. It also occasionally broke the `Action:` format, so I added `handle_parsing_errors=True`. In the final run every step parsed correctly; the agent sometimes repeated the `Question:` line in its reasoning, which was harmless.